# Hyperparameter Tuning (Ray Tune)

<div style="text-align: justify">

The following notebook is dedicated to <b>hyperparameter optimisation</b> for the <b>Tau Supersymmetry Anomaly Detection</b> analysis. Using <b>Ray Tune</b> with the <b>ASHA scheduler</b> for aggressive early stopping, a search is performed over the hyperparameter space for the autoencoder (AE) or variational autoencoder (VAE). Models are trained on <b>background only</b> using PyTorch Lightning, and the best configuration is exported as a Hydra-compatible YAML config.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis, model, and tuning configuration |
| Load | `io.load_dataframe` | Read mc.parquet from feature engineering output |
| Search Space | `tuning.build_search_space` | Build Ray Tune search space from config |
| Tune | `tuning.run_tune` | Run Ray Tune with ASHA scheduler and Lightning training |
| Results | `ray.tune` | Best trial summary and trial dataframe |
| Export | `tuning.export_best_config` | Save best config as Hydra-compatible YAML |

The same pipeline is available as a CLI via `uv run python run.py stage=tune`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)

Hyperparameter Optimisation:
* [Ray Tune](https://docs.ray.io/en/latest/tune/index.html)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
* [scikit-learn](https://scikit-learn.org/stable/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, model hyperparameters, tuning settings) are defined in `configs/` and can be overridden here.

> **Model selection:** change `overrides` to `["model=vae"]` for VAE tuning.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=ae"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]

models_dir.mkdir(parents=True, exist_ok=True)

## Data Loading & Preparation

Loading the MC DataFrame produced by the feature engineering pipeline.

In [ ]:
from src.processing.io import load_dataframe

df_mc = load_dataframe(dataframes_dir / "mc.parquet")
print(f"Loaded: {len(df_mc):,} events, {len(df_mc.columns)} columns")

## Search Space

Building the Ray Tune search space from the tuning config. The search space is defined in `configs/tuning/default.yaml` with separate entries for AE and VAE.

Separating training features and event weights from the MC DataFrame. For anomaly detection, models train on **background only** — signal events are held out for evaluation.

In [ ]:
from src.models.tuning import build_search_space

model_type = cfg.model.name
search_space = build_search_space(cfg.tuning.search_space[model_type])
print(f"Model type: {model_type}")
print(f"Search space parameters: {list(search_space.keys())}")

## Ray Tune Search

Running Ray Tune with the ASHA scheduler. Each trial trains an AE/VAE on the background-only train split using PyTorch Lightning and reports `val_loss` for early stopping.

In [ ]:
from src.models.tuning import run_tune

results = run_tune(
    cfg=cfg,
    df_mc=df_mc,
    search_space=search_space,
    num_samples=cfg.tuning.num_samples,
    model_type=model_type,
)
print(f"Tuning complete: {len(results)} trials")

## Results

### Best Trial Summary

In [ ]:
best_result = results.get_best_result(metric="val_loss", mode="min")

print(f"Best trial val_loss: {best_result.metrics['val_loss']:.6f}")
print(f"\nBest config:")
for k, v in best_result.config.items():
    print(f"  {k}: {v}")

### Trial Dataframe

Inspecting all trials as a DataFrame for further analysis.

In [ ]:
results_df = results.get_dataframe()
results_df.sort_values("val_loss").head(10)

## Export Best Parameters

Exporting the best trial's configuration as a Hydra-compatible YAML model config. This file can be copied to `configs/model/` and used for training:

```bash
cp <output_path> configs/model/ae_tuned.yaml
uv run python run.py stage=train model=ae_tuned
```

In [ ]:
from src.models.tuning import export_best_config

params_path = models_dir / f"{model_type}_best_params.yaml"
export_best_config(best_result, params_path)
print(f"Best params exported to: {params_path}")